In [6]:
import os
import random

from topic_gen.generate import TopicGenerator
from topic_gen import logger
from topic_gen.models import TopicList

logger.setLevel("INFO")

base_dir = "examples/data/"

person_map = {
    "birger": "Birger Larsen",
    "philipp": "Philipp Schaer",
    "timo": "Timo Breuer",
    "bjoern": "Björn Engelmann",
    "fabian": "Fabian Haag",
    "andreas": "Andreas Kruff",
    "jueri": "Jüri Keller"
}

experience_map = {
    "birger": 25,
    "philipp": 10,
    "timo": 6,
    "bjoern": 4,
    "fabian": 4,
    "andreas": 1,
    "jueri": 2
}

generator = TopicGenerator(
    model_name="gemini-2.5-flash",
    model_provider="google_genai",
    temperature=0.7,
)

metadata = []


for person in os.listdir(base_dir):
    output_path = os.path.join(base_dir, f"{person}/{person}-meta.csv")
    if os.path.exists(output_path):
        logger.info(f"Skipping {person}, already processed.")
        continue
    if person.startswith("."):
        continue
    if person.startswith("jueri_short"):
        continue
    res = []
    
    # base
    topics = generator.generate_topics(
        prompt_name="isearch-new-base",
        name=person_map[person],                        
        publication=os.path.join(base_dir, f"{person}/publication"),
        number_of_topics=5
    )
    for topic in topics.topics:
        res.append(topic)
        metadata.append({
            "topic_id": topic.id,
            "person": person_map[person],
            "prompt": "isearch-new-base",
        })
               
    
    
    # background
    topics = generator.generate_topics(
        prompt_name="isearch-background",
        name=person_map[person],        
        experience=experience_map[person],
        background=os.path.join(base_dir, f"{person}/background"),
        publication=os.path.join(base_dir, f"{person}/publication"),
        number_of_topics=5,
    )
    
    for topic in topics.topics:
        res.append(topic)
        metadata.append({
            "topic_id": topic.id,
            "person": person_map[person],
            "prompt": "isearch-background",
        })
    
    
    random.shuffle(res)

    topics = TopicList(topics=res)
    
    topics.to_xml(output=os.path.join(base_dir, f"{person}/{person}-topics.xml"))

    with open(output_path, "w") as f:
        f.write("topic_id,person,prompt\n")
        for m in metadata:
            f.write(f"{m['topic_id']},{m['person']},{m['prompt']}\n")


[topic_gen] [INFO] Skipping timo, already processed.
[topic_gen] [INFO] Skipping philipp, already processed.
[topic_gen] [INFO] Skipping bjoern, already processed.
[topic_gen] [INFO] Skipping birger, already processed.
[topic_gen] [INFO] Final text prompt:
 Objective:
You are to generate a set of realistic, task-based information needs for an IR test collection.

Your Persona:
You are, Jüri Keller, a research scientist. You will be given a list of your own publications. You must adopt this persona completely, writing in the first person ("I", "my project", "we are investigating") and grounding the information needs in your established research interests and expertise as evidenced by your publications.

Task:
Based on the provided list of your publications, create 5 new topics. These topics must represent plausible, real-world information needs you might have as you continue your research. Imagine scenarios like starting a new project, exploring a tangent from your previous work, prepar